# CSMODEL MCO — Student Sleep, Screen Time & Academic Performance

**Status: partial implementation (MCO1 foundation).** Code cells tagged `[AI-ASSISTED]` were drafted with AI and must be reviewed/understood by the group; cells tagged `[STUDENT TODO]` are for the group to write. All Markdown explanations must be written by the students.

## 1. Abstract

> **[STUDENTS TO WRITE]** Write after the analysis is done (research question, dataset, key findings).

## 2. Dataset Description

> **[STUDENTS TO WRITE]** Source, collection method and its implications, row/column meaning, attribute descriptions.
The dataset was obtained from Kaggle ("Student Sleep, Screen Time & Academic Performance" by Muhammad Aamir). It is important to note that the dataset is synthetically generated meaning it was not collected from real students. It was created programmatically using randomized sampling to simulate real world data. The creator of the dataset intentionally put human error to simulate real world messiness, like inconsistent capitalization in the gender column, missing values scattered across 5 columns at varying rates and a few injected outliers to help make it more realistic. The dataset contains 500 rows representing each student and 25 columns. Some key variables include academic performance, sleep metrics, screen time, and demographics.

In [1]:
import piplite
await piplite.install('seaborn')

In [2]:
# [AI-ASSISTED] Setup and loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "student_sleep_screen_time_academic_performance.csv"
raw = pd.read_csv(DATA_PATH)   # original, never modified
df = raw.copy()                # working copy used for all cleaning/analysis

print(raw.shape)
raw.dtypes

(500, 25)


student_id                       str
age                            int64
gender                           str
year_of_study                  int64
major_category                   str
residence_type                   str
chronotype                       str
avg_bedtime_hour             float64
avg_sleep_hours_weekday      float64
weekend_sleep_delta_hours    float64
pre_sleep_screen_time_min      int64
daily_screen_time_hours      float64
primary_device                   str
device_free_before_bed           str
caffeine_drinks_per_day        int64
naps_per_week                  int64
sleep_quality_score          float64
exercise_days_per_week         int64
energy_level                 float64
concentration_difficulty     float64
weekly_study_hours           float64
class_attendance_percent     float64
has_part_time_job                str
semester_type                    str
cgpa                         float64
dtype: object

In [3]:
# [AI-ASSISTED] Numerical summary and categorical value counts
display(raw.describe().T)
cat_cols = raw.select_dtypes(exclude="number").columns.drop("student_id")
for c in cat_cols:
    print(c, raw[c].value_counts(dropna=False).to_dict())

,count,mean,std,min,25%,50%,75%,max
age,500.0,20.260000,1.994079,17.00,19.000,20.00,22.000,27.00
year_of_study,500.0,2.954000,1.405644,1.00,2.000,3.00,4.000,5.00
avg_bedtime_hour,500.0,12.778200,10.772656,0.00,1.100,21.50,22.800,24.00
avg_sleep_hours_weekday,500.0,6.608200,1.435784,1.50,5.700,6.60,7.700,10.50
weekend_sleep_delta_hours,500.0,1.540600,0.965441,-1.00,0.900,1.50,2.200,4.40
pre_sleep_screen_time_min,500.0,38.388000,23.573478,0.00,22.000,35.50,54.000,240.00
daily_screen_time_hours,500.0,5.827000,2.046193,0.50,4.475,5.80,7.200,19.50
caffeine_drinks_per_day,500.0,1.502000,1.273279,0.00,1.000,1.00,2.000,7.00
naps_per_week,500.0,1.458000,1.331082,0.00,0.000,1.00,2.000,7.00
sleep_quality_score,494.0,7.271255,2.131409,1.00,6.000,7.00,9.000,10.00


gender {'Female': 190, 'Male': 180, 'male': 30, 'female': 28, 'F': 24, 'M': 23, 'Non-binary': 13, 'Prefer not to say': 12}
major_category {'STEM': 162, 'Business': 89, 'Social Sciences': 77, 'Health Sciences': 62, 'Humanities': 52, 'Arts': 33, 'Law': 25}
residence_type {'Dormitory': 190, 'Off-campus Apartment': 153, 'With Family': 133, 'Homestay': 24}
chronotype {'Intermediate': 201, 'Early Bird': 152, 'Night Owl': 147}
primary_device {'Smartphone': 271, 'Laptop': 198, 'Tablet': 31}
device_free_before_bed {'No': 368, 'Yes': 123, nan: 9}
has_part_time_job {'No': 331, 'Yes': 169}
semester_type {'Fall': 225, 'Spring': 201, 'Summer': 74}


## 3. Data Cleaning & Preprocessing

> **[STUDENTS TO WRITE]** Justify every cleaning decision (and every step you chose NOT to apply).
>
> ### Missing values and duplicates
> **Identification of Missing Data**: An initial assessment of the raw dataset revealed missing values distributed across five variables: device_free_before_bed (9 missing, 1.8%), sleep_quality_score (6 missing, 1.2%), energy_level (15 missing, 3.0%), concentration_difficulty (14 missing, 2.8%), and cgpa (7 missing, 1.4%). Because these missing entries represent less than 3.5% of total observations per column, they fall within an acceptable threshold where standard imputation or selective dropping can be applied without introducing severe bias.
> 
> **Uniqueness and Duplicate Checking**: Both fully duplicated rows and duplicate student_id entries were evaluated to ensure record uniqueness. The checks confirmed 0 fully duplicated rows and 0 duplicate student IDs, verifying that each record in the dataset corresponds to a unique student participant. Consequently, no row-level deduplication is required at this stage.
>
> ### IQR outlier screen
> **Methodology Selection**: To detect potential anomalies without prematurely discarding information, the Interquartile Range (IQR) method was applied using the standard $1.5 \times \text{IQR}$ rule across key numerical columns: avg_sleep_hours_weekday, daily_screen_time_hours, pre_sleep_screen_time_min, weekly_study_hours, class_attendance_percent, and cgpa
>
> **Conservative Flagging Strategy**: Following a non-destructive approach, values falling outside the lower ($Q1 - 1.5 \times \text{IQR}$) and upper ($Q3 + 1.5 \times \text{IQR}$) fences were flagged rather than automatically removed. This ensures that extreme behavioral variations (such as exceptionally high study hours or screen times) are preserved, as they may represent valid behavioral extremes among students rather than data collection artifacts.
>
> ### Inspect avg_bedtime_hour encoding
> **Inspection Purpose**: To understand how student sleep schedules and post-midnight bedtimes are numerically represented, a descriptive statistical analysis (describe()) was conducted on the avg_bedtime_hour feature.
>
> **Distribution Analysis**: The summary statistics across all 500 records show a minimum value of $0.00$ and a maximum value of $24.00$, indicating a continuous 24-hour time scale format. The median ($50\%$) value of $21.50$ (9:30 PM) combined with a first quartile ($25\%$) of $1.10$ (approx. 1:06 AM) demonstrates that student sleep patterns span across midnight, capturing both early and late sleepers.
>
> **Completeness**: The count of $500.00$ confirms that there are no missing entries in this feature, ensuring high data integrity and reliability for subsequent sleep behavior analyses.
>
> ### Cleaning on the working copy: label consistency + bedtime wrap-around
> **Categorical Label Standardization**: To correct inconsistencies and formatting variants in the gender column (such as 'male', 'M', and 'Male'), values were mapped and standardized into unified categories (Female, Male). As shown in the post-cleaning value counts, the distribution across the dataset comprises 242 Female, 233 Male, 13 Non-binary, and 12 Prefer not to say respondents. This standardization prevents fragmentation during grouping, aggregation, and categorical modeling.
>
> **Bedtime Feature Engineering (Midnight Wrap-Around Adjustment)**: Because bedtimes occurring after midnight were natively recorded on a 0–6 hour scale, treating them strictly numerically would distort mathematical models by making late-night sleep look artificially early (e.g., 1:00 AM appearing smaller than 10:00 PM). To resolve this, a continuous shifted scale (bedtime_shifted) was engineered by adding 24 hours to any bedtime values below 12 (e.g., shifting 1:00 AM to 25.0). This maintains chronological continuity for regression and statistical correlation analyses.

In [4]:
# [AI-ASSISTED] Missing values and duplicates
missing = raw.isna().sum()
display(missing[missing > 0].to_frame("n_missing").assign(pct=lambda t: (t.n_missing / len(raw) * 100).round(1)))
print("Fully duplicated rows:", raw.duplicated().sum())
print("Duplicated student_id:", raw["student_id"].duplicated().sum())

,n_missing,pct
device_free_before_bed,9,1.8
sleep_quality_score,6,1.2
energy_level,15,3.0
concentration_difficulty,14,2.8
cgpa,7,1.4


Fully duplicated rows: 0
Duplicated student_id: 0


In [5]:
# [AI-ASSISTED] IQR outlier screen (flags only; nothing is removed here)
num_cols = ["avg_sleep_hours_weekday", "daily_screen_time_hours", "pre_sleep_screen_time_min",
            "weekly_study_hours", "class_attendance_percent", "cgpa"]
rows = []
for c in num_cols:
    s = raw[c].dropna()
    q1, q3 = s.quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    rows.append((c, round(lo, 2), round(hi, 2), int(((s < lo) | (s > hi)).sum()), s.min(), s.max()))
pd.DataFrame(rows, columns=["variable", "lower_fence", "upper_fence", "n_flagged", "min", "max"])

,variable,lower_fence,upper_fence,n_flagged,min,max
0,avg_sleep_hours_weekday,2.70,10.70,2,1.50,10.50
1,daily_screen_time_hours,0.39,11.29,2,0.50,19.50
2,pre_sleep_screen_time_min,-26.00,102.00,3,0.00,240.00
3,weekly_study_hours,-0.24,32.66,3,0.00,88.50
4,class_attendance_percent,45.05,113.65,3,34.60,100.00
5,cgpa,1.98,3.69,6,1.89,3.95


In [6]:
# [AI-ASSISTED] Inspect avg_bedtime_hour encoding (check how post-midnight bedtimes are stored)
raw["avg_bedtime_hour"].describe()

count    500.000000
mean      12.778200
std       10.772656
min        0.000000
25%        1.100000
50%       21.500000
75%       22.800000
max       24.000000
Name: avg_bedtime_hour, dtype: float64

In [7]:
# [AI-ASSISTED] Cleaning on the working copy: label consistency + bedtime wrap-around
# Gender has variants of the same label (e.g. 'male', 'M', 'Male').
df["gender"] = df["gender"].replace({"male": "Male", "M": "Male", "female": "Female", "F": "Female"})

# Bedtimes after midnight are stored as 0-6; shift by +24 so 1am becomes 25.0 (continuous scale).
df["bedtime_shifted"] = np.where(df["avg_bedtime_hour"] < 12, df["avg_bedtime_hour"] + 24, df["avg_bedtime_hour"])
df["gender"].value_counts()

gender
Female               242
Male                 233
Non-binary            13
Prefer not to say     12
Name: count, dtype: int64

In [8]:
# [STUDENT TODO] Decide and implement handling of missing values
# (cgpa, sleep_quality_score, energy_level, concentration_difficulty, device_free_before_bed)
# and of the flagged outliers. Justify in the Markdown above.
df = df.dropna(subset=["cgpa"]).copy()

score_cols = ["sleep_quality_score", "energy_level", "concentration_difficulty"]
df[score_cols] = df[score_cols].fillna(df[score_cols].median())

df["device_free_before_bed"] = df["device_free_before_bed"].fillna("Unknown")

impossible = (df["daily_screen_time_hours"] + df["avg_sleep_hours_weekday"] > 24) | (df["weekly_study_hours"] > 84)
display(df.loc[impossible, ["student_id", "daily_screen_time_hours", "avg_sleep_hours_weekday", "weekly_study_hours"]])
df = df[~impossible].copy()

print("Rows kept:", len(df), "of", len(raw), "| remaining missing cells:", int(df.isna().sum().sum()))

,student_id,daily_screen_time_hours,avg_sleep_hours_weekday,weekly_study_hours
449,STU2450,19.5,4.6,17.7
462,STU2463,7.4,6.7,88.5


Rows kept: 491 of 500 | remaining missing cells: 0


In [9]:
# [STUDENT TODO] Feature engineering for later sections, e.g. sleep-duration groups, screen-time groups
df["sleep_group"] = pd.cut(df["avg_sleep_hours_weekday"], bins=[0, 6, 8, 24], right=False,
                           labels=["Short (<6h)", "Moderate (6-8h)", "Long (8h+)"])

df["screen_group"] = pd.qcut(df["daily_screen_time_hours"], q=3, labels=["Low", "Medium", "High"])

df["pre_sleep_group"] = pd.cut(df["pre_sleep_screen_time_min"], bins=[-1, 30, 60, np.inf],
                               labels=["0-30 min", "31-60 min", "60+ min"])

for c in ["sleep_group", "screen_group", "pre_sleep_group"]:
    print(c, df[c].value_counts().sort_index().to_dict())

sleep_group {'Short (<6h)': 152, 'Moderate (6-8h)': 244, 'Long (8h+)': 95}
screen_group {'Low': 167, 'Medium': 164, 'High': 160}
pre_sleep_group {'0-30 min': 197, '31-60 min': 214, '60+ min': 80}


## 4. Research Question & Exploratory Data Analysis

> **[STUDENTS TO WRITE]** State the general research question and how you refined it. Max 15 figures in the whole notebook.

### EDA Question 1

> **[STUDENTS TO WRITE]** State the question.

In [10]:
# [STUDENT TODO] EDA Q1: numerical summary + appropriate visualization


### EDA Question 2

> **[STUDENTS TO WRITE]** State the question.

In [11]:
# [STUDENT TODO] EDA Q2: numerical summary + appropriate visualization


### EDA Question 3

> **[STUDENTS TO WRITE]** State the question.

In [12]:
# [STUDENT TODO] EDA Q3: numerical summary + appropriate visualization


## 5. Data Mining

> **[RESERVED FOR MCO2]** Rule mining, clustering, or collaborative filtering.

## 6. Statistical Inference

> **[RESERVED FOR MCO2]** At least two tests with hypotheses, assumption checks, p-values and significance level.

## 7. Summary, Conclusions, and Recommendations

> **[RESERVED FOR MCO2]**

## 8. References & Disclosures

> **[STUDENTS TO WRITE]** List references and add the generative-AI disclosure statement in the format required by the specs (tool name, purposes of use, review statement).